# Fink/LSST — Reload and Display Saved Light Curves

This notebook reads the Parquet files saved by `01_fink_block_lightcurves.ipynb`  
and reproduces the same visualisations (flux and magnitude) per source group.

**Expected data** in `data_FINK_BLOCK_LC_01/`:
- `{group}_fp.parquet`  — forced-photometry light curves
- `{group}_src.parquet` — detection-based light curves (diaSources)
- `flatness_metrics.csv` — pre-computed flatness metrics

No API call is made in this notebook.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from astropy.time import Time
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")
from scipy.signal import lombscargle

In [ ]:
# Remove to run faster the notebook
# import ipywidgets as widgets
# %matplotlib widget

# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
inputfile = "data_FINK_BLOCK_LC_01/gaia_nophotgstar_stable_unknown_parallax_src.parquet"

In [ ]:
diaobjectid = 313875416704090123

In [ ]:
df = pd.read_parquet(inputfile)

In [ ]:
df

In [ ]:
df = df[df["r:diaObjectId"] == diaobjectid]

In [ ]:
df.columns

In [ ]:
P = 8.28

In [ ]:
df["tphase"] = df["r:midpointMjdTai"] - np.floor(df["r:midpointMjdTai"] / P) * P

In [ ]:
listbands = "ugrizy"


dict_band = {}
for b in listbands:
    print(b)
    dict_band[b] = df[df["r:band"] == b]

dict_color = {"u": "b", "g": "g", "r": "r", "i": "orange", "z": "brown", "y": "grey"}

In [ ]:
dict_band["g"]

In [ ]:
fig, axs = plt.subplots(6, 1, figsize=(10, 10), sharex=True)

for idx, b in enumerate(listbands):
    ax = axs[idx]
    dfb = dict_band[b]
    color = dict_color[b]
    dfb.plot(x="tphase", y="r:psfFlux", ax=ax, lw=0, marker="+", color=color)
    ax.grid()

In [ ]:
t = df["r:midpointMjdTai"].values
bands = df["r:band"].values
y = df["r:psfFlux"].values
dy = df["r:psfFluxErr"].values

In [ ]:
from astropy.timeseries import LombScargleMultiband

frequency, power = LombScargleMultiband(t, y, bands, dy).autopower()

In [ ]:
frequency.shape

In [ ]:
power.shape

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

ax.plot(1 / frequency, power, marker="+", lw=0)
ax.grid()
plt.show()

In [ ]:
LombScargleMultiband?

In [ ]:
power